# 🎯 Grounding DINO + SAM Practice Notebook
## Open-Set Detection & Segmentation — find with text, cut at pixel level

This notebook is a follow-up to **Lab 1 (DINO ViT practice)**.
Using the same Oxford-IIIT Pet data, we will practice the following:

1. **Grounding DINO** — open-set object detection with text prompts
2. **SAM (Segment Anything Model)** — bounding box → high-quality segmentation mask
3. **Grounded SAM pipeline** — combining detection + segmentation
4. **Experiments with various text prompts** — comparing results across prompts
5. **Application: background removal** — a practical example
6. **Threshold experiments** — precision vs recall trade-off
7. **SAM's multiple masks** — how ambiguity is handled

### Core Idea
```
"dog" (text) ──→ [Grounding DINO] ──→ Bounding Boxes ──→ [SAM] ──→ Pixel-level Masks
```

---

## 0. Environment Setup

In [1]:
!pip install transformers torch torchvision accelerate Pillow matplotlib numpy scikit-learn tqdm requests scipy -q

In [2]:
import os, random, requests, tarfile, shutil
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from PIL import Image
from tqdm.notebook import tqdm

import torch
import torch.nn.functional as F
from transformers import (
    AutoProcessor, AutoModelForZeroShotObjectDetection,
    SamModel, SamProcessor,
)
import warnings
warnings.filterwarnings('ignore')

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"🖥️ Using device: {device}")

🖥️ Using device: cpu


---
## 1. Load Models

- **Grounding DINO**: [Liu et al., 2023](https://arxiv.org/abs/2303.05499) — detects any object via text
- **SAM**: [Kirillov et al., 2023](https://arxiv.org/abs/2304.02643) — a foundation model trained on 11M images and 1B+ masks

In [3]:
GDINO_ID = "IDEA-Research/grounding-dino-tiny"
print(f"📦 Grounding DINO: {GDINO_ID}")
gdino_processor = AutoProcessor.from_pretrained(GDINO_ID)
gdino_model = AutoModelForZeroShotObjectDetection.from_pretrained(GDINO_ID).to(device)
gdino_model.eval()
gdino_params = sum(p.numel() for p in gdino_model.parameters())
print(f"   ✅ Done! ({gdino_params/1e6:.1f}M params)")

📦 Grounding DINO: IDEA-Research/grounding-dino-tiny


preprocessor_config.json:   0%|          | 0.00/457 [00:00<?, ?B/s]

The image processor of type `GroundingDinoImageProcessor` is now loaded as a fast processor by default, even if the model checkpoint was saved with a slow processor. This is a breaking change and may produce slightly different outputs. To continue using the slow processor, instantiate this class with `use_fast=False`. 


config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

...

added_tokens.json:   0%|          | 0.00/82.0 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/689M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/990 [00:00<?, ?it/s]

   ✅ 완료! (172.2M params)


In [4]:
SAM_ID = "facebook/sam-vit-base"
print(f"📦 SAM: {SAM_ID}")
sam_processor = SamProcessor.from_pretrained(SAM_ID)
sam_model = SamModel.from_pretrained(SAM_ID).to(device)
sam_model.eval()
sam_params = sum(p.numel() for p in sam_model.parameters())
print(f"   ✅ Done! ({sam_params/1e6:.1f}M params)")
print(f"\n📊 Total: {(gdino_params + sam_params)/1e6:.1f}M params")

📦 SAM: facebook/sam-vit-base


preprocessor_config.json:   0%|          | 0.00/466 [00:00<?, ?B/s]

The image processor of type `SamImageProcessor` is now loaded as a fast processor by default, even if the model checkpoint was saved with a slow processor. This is a breaking change and may produce slightly different outputs. To continue using the slow processor, instantiate this class with `use_fast=False`. 


config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/375M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/314 [00:00<?, ?it/s]

   ✅ 완료! (93.7M params)

📊 Total: 266.0M params


---
## 2. Load Oxford-IIIT Pet Data

Same dataset as Lab 1. If the `./data` folder exists, the cache is used.

In [5]:
NUM_IMAGES_PER_CLASS = 50
IMG_DIR = './data'; CACHE_DIR = './cache'
for d in [IMG_DIR, CACHE_DIR, f'{IMG_DIR}/dog', f'{IMG_DIR}/cat']:
    os.makedirs(d, exist_ok=True)

def download_oxford_pets():
    extract_dir = os.path.join(CACHE_DIR, 'images')
    if os.path.exists(extract_dir) and len(os.listdir(extract_dir)) > 100:
        print('   Using cache'); return extract_dir
    url = 'https://www.robots.ox.ac.uk/~vgg/data/pets/data/images.tar.gz'
    tar_path = os.path.join(CACHE_DIR, 'images.tar.gz')
    print(f'   Downloading: {url} (about 800MB)')
    r = requests.get(url, stream=True, timeout=120)
    total = int(r.headers.get('content-length', 0))
    with open(tar_path, 'wb') as f:
        dl = 0
        for c in r.iter_content(1024*1024):
            f.write(c); dl += len(c)
            if total: print(f'\r   {dl/(1024*1024):.0f}/{total/(1024*1024):.0f}MB', end='')
    print('\n   Extracting archive...')
    with tarfile.open(tar_path, 'r:gz') as t: t.extractall(CACHE_DIR)
    return extract_dir

def organize_dog_cat(src, dst, n):
    dogs = [f for f in sorted(os.listdir(src)) if f[0].islower() and f.endswith(('.jpg','.jpeg','.png'))]
    cats = [f for f in sorted(os.listdir(src)) if f[0].isupper() and f.endswith(('.jpg','.jpeg','.png'))]
    random.shuffle(dogs); random.shuffle(cats)
    for i,f in enumerate(dogs[:n]): Image.open(f'{src}/{f}').convert('RGB').save(f'{dst}/dog/dog_{i:03d}.jpg')
    for i,f in enumerate(cats[:n]): Image.open(f'{src}/{f}').convert('RGB').save(f'{dst}/cat/cat_{i:03d}.jpg')

ed = len([f for f in os.listdir(f'{IMG_DIR}/dog') if f.endswith('.jpg')])
ec = len([f for f in os.listdir(f'{IMG_DIR}/cat') if f.endswith('.jpg')])
if ed >= NUM_IMAGES_PER_CLASS and ec >= NUM_IMAGES_PER_CLASS:
    print(f'✅ Data ready: Dogs={ed}, Cats={ec}')
else:
    print('📥 Downloading...'); organize_dog_cat(download_oxford_pets(), IMG_DIR, NUM_IMAGES_PER_CLASS)

image_paths, image_labels = [], []
for label in ['dog', 'cat']:
    for f in sorted(os.listdir(f'{IMG_DIR}/{label}')):
        if f.endswith('.jpg'): image_paths.append(f'{IMG_DIR}/{label}/{f}'); image_labels.append(label)
print(f"📂 Total {len(image_paths)} images: Dog {image_labels.count('dog')}, Cat {image_labels.count('cat')}")

📥 다운로드...
   다운로드: https://www.robots.ox.ac.uk/~vgg/data/pets/data/images.tar.gz (약 800MB)
   755/755MB
   압축 해제 중...
📂 총 100장: Dog 50, Cat 50


---
## 3. Grounding DINO — Object Detection with Text Prompts

| | Traditional (YOLO, RCNN) | Grounding DINO |
|---|---|---|
| Classes | Fixed at training time (80 classes) | Any text |
| Prompt | None | Natural-language text |
| Zero-shot | ❌ | ✅ |

In [6]:
@torch.no_grad()
def detect_objects(image, text_labels, box_threshold=0.3, text_threshold=0.25):
    """
    Use Grounding DINO to detect objects matching the text in an image.

    Args:
        image: PIL Image
        text_labels: list of str (e.g., ["dog", "cat"])
    Returns:
        boxes (N,4), scores (N,), labels list[str]
    """
    inputs = gdino_processor(
        images=image, text=[text_labels], return_tensors="pt"
    ).to(device)
    outputs = gdino_model(**inputs)
    results = gdino_processor.post_process_grounded_object_detection(
        outputs, threshold=box_threshold, text_threshold=text_threshold,
        target_sizes=[(image.height, image.width)]
    )
    r = results[0]
    return r['boxes'].cpu().numpy(), r['scores'].cpu().numpy(), r['text_labels']

print("✅ detect_objects defined")

✅ detect_objects 정의 완료


In [7]:
def draw_detections(image, boxes, scores, labels, title='', ax=None):
    if ax is None: fig, ax = plt.subplots(1, 1, figsize=(8, 8))
    ax.imshow(image)
    unique = list(set(labels))
    cmap = plt.cm.Set2
    cdict = {l: cmap(i / max(len(unique), 1)) for i, l in enumerate(unique)}
    for box, score, label in zip(boxes, scores, labels):
        x1, y1, x2, y2 = box
        c = cdict[label]
        rect = mpatches.FancyBboxPatch((x1,y1), x2-x1, y2-y1, linewidth=3,
            edgecolor=c, facecolor='none', boxstyle='round,pad=0')
        ax.add_patch(rect)
        ax.text(x1, y1-8, f'{label}: {score:.2f}', fontsize=11, fontweight='bold',
                color='white', bbox=dict(boxstyle='round,pad=0.3', facecolor=c, edgecolor='none', alpha=0.85))
    ax.set_title(title, fontsize=13, fontweight='bold'); ax.axis('off')
    return ax

print("✅ draw_detections defined")

✅ draw_detections 정의 완료


In [8]:
# Test Grounding DINO on Dog/Cat images
sample_indices = [0, 1, NUM_IMAGES_PER_CLASS, NUM_IMAGES_PER_CLASS + 1]
fig, axes = plt.subplots(2, 2, figsize=(14, 14))
fig.suptitle('🎯 Grounding DINO — detecting "dog" and "cat"', fontsize=16, fontweight='bold')

for i, idx in enumerate(sample_indices):
    ax = axes[i // 2, i % 2]
    img = Image.open(image_paths[idx]).convert('RGB')
    boxes, scores, labels = detect_objects(img, ["dog", "cat"], box_threshold=0.25)
    gt = image_labels[idx].upper()
    draw_detections(img, boxes, scores, labels,
                   title=f'GT: {gt} | {len(boxes)} detections', ax=ax)

plt.tight_layout(); plt.show()

Output hidden; open in https://colab.research.google.com to view.

### Comparing Various Text Prompts

The strength of Grounding DINO: change the prompt and you detect different things!

In [ ]:
sample_img = Image.open(image_paths[0]).convert('RGB')

prompt_experiments = [
    (["dog"],                   '"dog"'),
    (["animal"],                '"animal"'),
    (["golden retriever"],      '"golden retriever"'),
    (["head", "body", "leg"],   '"head", "body", "leg"'),
]

fig, axes = plt.subplots(1, len(prompt_experiments), figsize=(6*len(prompt_experiments), 6))
fig.suptitle('🔍 Same image, different prompts → different detection results', fontsize=16, fontweight='bold')

for i, (text_labels, name) in enumerate(prompt_experiments):
    boxes, scores, labels = detect_objects(sample_img, text_labels, box_threshold=0.2)
    draw_detections(sample_img, boxes, scores, labels,
                   title=f'Prompt: {name}\n({len(boxes)} detections)', ax=axes[i])

plt.tight_layout(); plt.show()

---
## 4. SAM — Segment Anything Model

SAM generates segmentation masks from various prompts:
- **Point prompt**: click a specific coordinate
- **Box prompt**: bounding box ← connects to Grounding DINO output!

In [10]:
@torch.no_grad()
def segment_with_boxes(image, boxes):
    """
    Pass bounding boxes to SAM as prompts to generate segmentation masks.
    Process boxes one at a time to ensure shape stability.

    Args:
        image: PIL Image
        boxes: numpy array (N, 4) [x1, y1, x2, y2]
    Returns:
        masks: list of (H, W) bool numpy arrays
        iou_scores: list of float
    """
    if len(boxes) == 0:
        return [], []

    best_masks = []
    best_scores = []

    for box in boxes:
        # SAM processor expects: input_boxes = [[[x1, y1, x2, y2]]]
        input_boxes = [[box.tolist()]]
        inputs = sam_processor(
            images=image, input_boxes=input_boxes, return_tensors="pt"
        ).to(device)

        outputs = sam_model(**inputs)

        # post_process_masks: restore to original image size
        masks_list = sam_processor.image_processor.post_process_masks(
            outputs.pred_masks.cpu(),
            inputs["original_sizes"].cpu(),
            inputs["reshaped_input_sizes"].cpu()
        )
        iou = outputs.iou_scores.cpu().numpy()  # (1, 1, 3)

        # masks_list[0] shape: (1, 3, H, W) → squeeze → (3, H, W)
        mask_3 = masks_list[0].squeeze(0)   # (3, H, W)
        score_3 = iou[0, 0]                  # (3,)

        best_idx = score_3.argmax()
        best_masks.append(mask_3[best_idx].numpy().astype(bool))
        best_scores.append(float(score_3[best_idx]))

    return best_masks, best_scores


@torch.no_grad()
def segment_with_points(image, points, point_labels=None):
    """Pass point prompts to SAM to generate a segmentation mask."""
    if point_labels is None:
        point_labels = [1] * len(points)

    inputs = sam_processor(
        images=image, input_points=[[points]], input_labels=[[point_labels]],
        return_tensors="pt"
    ).to(device)

    outputs = sam_model(**inputs)
    masks_list = sam_processor.image_processor.post_process_masks(
        outputs.pred_masks.cpu(),
        inputs["original_sizes"].cpu(),
        inputs["reshaped_input_sizes"].cpu()
    )
    iou = outputs.iou_scores.cpu().numpy()
    mask_3 = masks_list[0].squeeze(0)  # (3, H, W)
    score_3 = iou[0, 0]                # (3,)
    best_idx = score_3.argmax()
    return mask_3[best_idx].numpy().astype(bool), float(score_3[best_idx])

print("✅ SAM segmentation functions defined")

✅ SAM segmentation 함수 정의 완료


### SAM — Point Prompt Visualization

In [ ]:
sample_img = Image.open(image_paths[1]).convert('RGB')
w, h = sample_img.size
center_point = [[w // 2, h // 2]]

mask, iou = segment_with_points(sample_img, center_point)

fig, axes = plt.subplots(1, 3, figsize=(18, 6))
fig.suptitle('SAM — Point Prompt Segmentation', fontsize=16, fontweight='bold')

axes[0].imshow(sample_img)
axes[0].scatter([center_point[0][0]], [center_point[0][1]],
               c='red', s=200, marker='*', zorder=5, edgecolors='white', linewidth=2)
axes[0].set_title('Input: Image + Point'); axes[0].axis('off')

axes[1].imshow(mask, cmap='coolwarm')
axes[1].set_title(f'Output: Mask (IoU: {iou:.3f})'); axes[1].axis('off')

axes[2].imshow(sample_img)
overlay = np.zeros((*mask.shape, 4)); overlay[mask] = [0, 0.7, 1, 0.5]
axes[2].imshow(overlay)
axes[2].scatter([center_point[0][0]], [center_point[0][1]],
               c='red', s=200, marker='*', zorder=5, edgecolors='white', linewidth=2)
axes[2].set_title('Overlay'); axes[2].axis('off')

plt.tight_layout(); plt.show()

---
## 5. Grounded SAM Pipeline

```
Image + "dog" ──→ [Grounding DINO] ──→ Boxes ──→ [SAM] ──→ Masks
```

In [12]:
def grounded_sam(image, text_labels, box_threshold=0.3, text_threshold=0.25):
    """Grounded SAM: Detection → Segmentation"""
    boxes, scores, labels = detect_objects(image, text_labels, box_threshold, text_threshold)
    if len(boxes) == 0:
        return boxes, scores, labels, [], []
    masks, ious = segment_with_boxes(image, boxes)
    return boxes, scores, labels, masks, ious

print("✅ grounded_sam pipeline defined")

✅ grounded_sam 파이프라인 정의 완료


In [13]:
def visualize_grounded_sam(image, boxes, scores, labels, masks, title=''):
    """Original / Detection / Mask / Segmented — 4-panel visualization"""
    from scipy import ndimage
    fig, axes = plt.subplots(1, 4, figsize=(24, 6))
    img_arr = np.array(image)

    axes[0].imshow(image); axes[0].set_title('Original', fontweight='bold'); axes[0].axis('off')
    draw_detections(image, boxes, scores, labels,
                   title=f'Grounding DINO ({len(boxes)} boxes)', ax=axes[1])

    axes[2].imshow(image)
    colors = plt.cm.Set2(np.linspace(0, 1, max(len(masks), 1)))
    for i, m in enumerate(masks):
        c = colors[i % len(colors)]
        rgba = np.zeros((*m.shape, 4)); rgba[m] = [c[0], c[1], c[2], 0.5]
        axes[2].imshow(rgba)
        edge = ndimage.binary_dilation(m, iterations=2) ^ m
        edge_rgba = np.zeros((*m.shape, 4)); edge_rgba[edge] = [c[0], c[1], c[2], 1.0]
        axes[2].imshow(edge_rgba)
    axes[2].set_title(f'SAM ({len(masks)} masks)', fontweight='bold'); axes[2].axis('off')

    if masks:
        combined = np.any(np.stack(masks), axis=0)
        seg = img_arr.copy(); seg[~combined] = 255
        axes[3].imshow(seg)
    else:
        axes[3].imshow(image)
    axes[3].set_title('Segmented', fontweight='bold'); axes[3].axis('off')

    fig.suptitle(title, fontsize=14, fontweight='bold')
    plt.tight_layout(); plt.show()

print("✅ visualize_grounded_sam defined")

✅ visualize_grounded_sam 정의 완료


In [14]:
print("🚀 Running the Grounded SAM pipeline\n")

demo_indices = [0, 3, NUM_IMAGES_PER_CLASS, NUM_IMAGES_PER_CLASS + 3]
for idx in demo_indices:
    img = Image.open(image_paths[idx]).convert('RGB')
    gt = image_labels[idx]
    boxes, scores, labels, masks, ious = grounded_sam(img, [gt], box_threshold=0.25)
    visualize_grounded_sam(img, boxes, scores, labels, masks,
        title=f'Prompt: "{gt}" | GT: {gt.upper()}')

Output hidden; open in https://colab.research.google.com to view.

### Open-Vocabulary: Grounded SAM with Various Prompts

In [ ]:
sample_img = Image.open(image_paths[0]).convert('RGB')

creative_prompts = [
    (["dog"],                   '"dog"'),
    (["head", "body"],          '"head", "body"'),
    (["furry animal"],          '"furry animal"'),
    (["pet", "background"],     '"pet", "background"'),
]

fig, axes = plt.subplots(2, len(creative_prompts), figsize=(6*len(creative_prompts), 12))
fig.suptitle('🧪 Open-Vocabulary Grounded SAM', fontsize=16, fontweight='bold')

for col, (text_labels, name) in enumerate(creative_prompts):
    boxes, scores, labels, masks, ious = grounded_sam(sample_img, text_labels, box_threshold=0.2)
    draw_detections(sample_img, boxes, scores, labels,
                   title=f'Prompt: {name}\n{len(boxes)} detections', ax=axes[0, col])
    axes[1, col].imshow(sample_img)
    if masks:
        colors = plt.cm.Set2(np.linspace(0, 1, max(len(masks), 1)))
        for i, m in enumerate(masks):
            rgba = np.zeros((*m.shape, 4))
            rgba[m] = [colors[i][0], colors[i][1], colors[i][2], 0.5]
            axes[1, col].imshow(rgba)
    axes[1, col].set_title(f'{len(masks)} masks'); axes[1, col].axis('off')

plt.tight_layout(); plt.show()

---
## 6. Application: Background Removal

In [ ]:
n_demo = 6
demo_paths = random.sample(image_paths, n_demo)
fig, axes = plt.subplots(3, n_demo, figsize=(4*n_demo, 12))
fig.suptitle('✂️ Grounded SAM — Automatic Background Removal', fontsize=16, fontweight='bold')

for col, path in enumerate(demo_paths):
    img = Image.open(path).convert('RGB')
    gt = 'dog' if 'dog' in path else 'cat'
    img_arr = np.array(img)
    boxes, scores, labels, masks, ious = grounded_sam(img, [gt], box_threshold=0.25)

    axes[0, col].imshow(img); axes[0, col].set_title(gt.upper()); axes[0, col].axis('off')

    axes[1, col].imshow(img)
    if masks:
        all_m = np.any(np.stack(masks), axis=0)
        rgba = np.zeros((*all_m.shape, 4)); rgba[all_m] = [0, 0.8, 0.4, 0.4]
        axes[1, col].imshow(rgba)
    axes[1, col].set_title('Mask'); axes[1, col].axis('off')

    if masks:
        all_m = np.any(np.stack(masks), axis=0)
        checker = np.zeros((*img_arr.shape[:2], 3), dtype=np.uint8)
        blk = 20
        for r in range(0, checker.shape[0], blk):
            for c in range(0, checker.shape[1], blk):
                v = 200 if (r//blk + c//blk) % 2 == 0 else 255
                checker[r:r+blk, c:c+blk] = v
        alpha = all_m[:,:,np.newaxis].astype(float)
        result = (checker * (1 - alpha) + img_arr * alpha).astype(np.uint8)
        axes[2, col].imshow(result)
    else:
        axes[2, col].imshow(img)
    axes[2, col].set_title('BG Removed'); axes[2, col].axis('off')

plt.tight_layout(); plt.show()

---
## 7. Threshold Experiment

Changing `box_threshold` changes detection sensitivity.
Lower → recall↑ precision↓; higher → precision↑ recall↓

In [ ]:
sample_img = Image.open(image_paths[0]).convert('RGB')
thresholds = [0.1, 0.2, 0.3, 0.5, 0.7]

fig, axes = plt.subplots(2, len(thresholds), figsize=(5*len(thresholds), 10))
fig.suptitle('⚙️ Effect of changing the box threshold', fontsize=16, fontweight='bold')

for col, thr in enumerate(thresholds):
    boxes, scores, labels, masks, ious = grounded_sam(sample_img, ["dog", "cat"], box_threshold=thr)
    draw_detections(sample_img, boxes, scores, labels,
                   title=f'thr={thr} | {len(boxes)} boxes', ax=axes[0, col])
    axes[1, col].imshow(sample_img)
    if masks:
        colors = plt.cm.Set2(np.linspace(0, 1, max(len(masks), 1)))
        for i, m in enumerate(masks):
            rgba = np.zeros((*m.shape, 4))
            rgba[m] = [colors[i][0], colors[i][1], colors[i][2], 0.5]
            axes[1, col].imshow(rgba)
    axes[1, col].set_title(f'{len(masks)} masks'); axes[1, col].axis('off')

plt.tight_layout(); plt.show()

---
## 8. SAM's Multi-Mask Output

SAM produces **3 masks** for a single prompt (one per granularity).
It automatically selects the one with the highest IoU score.

In [ ]:
sample_img = Image.open(image_paths[0]).convert('RGB')
boxes, scores, labels = detect_objects(sample_img, ["dog"], box_threshold=0.3)

if len(boxes) > 0:
    single_box = [[boxes[0].tolist()]]
    inputs = sam_processor(images=sample_img, input_boxes=single_box, return_tensors="pt").to(device)

    with torch.no_grad():
        outputs = sam_model(**inputs)

    masks_list = sam_processor.image_processor.post_process_masks(
        outputs.pred_masks.cpu(),
        inputs["original_sizes"].cpu(),
        inputs["reshaped_input_sizes"].cpu()
    )
    iou_scores = outputs.iou_scores.cpu().numpy()  # (1, 1, 3)
    three_masks = masks_list[0].squeeze(0)          # (3, H, W)
    scores_3 = iou_scores[0, 0]                     # (3,)

    fig, axes = plt.subplots(1, 4, figsize=(20, 5))
    fig.suptitle('🎭 SAM — One Box → 3 masks (different granularities)', fontsize=14, fontweight='bold')

    axes[0].imshow(sample_img)
    x1, y1, x2, y2 = boxes[0]
    rect = mpatches.Rectangle((x1,y1), x2-x1, y2-y1, linewidth=3, edgecolor='lime', facecolor='none')
    axes[0].add_patch(rect)
    axes[0].set_title('Input Box'); axes[0].axis('off')

    mask_names = ['Mask 1 (smallest)', 'Mask 2 (medium)', 'Mask 3 (largest)']
    cs = [[1,0.3,0.3,0.5], [0.3,1,0.3,0.5], [0.3,0.3,1,0.5]]
    for i in range(3):
        m = three_masks[i].numpy()
        axes[i+1].imshow(sample_img)
        ov = np.zeros((*m.shape, 4)); ov[m > 0] = cs[i]
        axes[i+1].imshow(ov)
        star = ' ⭐' if i == scores_3.argmax() else ''
        axes[i+1].set_title(f'{mask_names[i]}\nIoU: {scores_3[i]:.3f}{star}')
        axes[i+1].axis('off')

    plt.tight_layout(); plt.show()
    print("💡 SAM produces 3 candidate masks to handle ambiguity.")
    print("   The mask with the highest IoU score (⭐) is selected automatically.")
else:
    print("No boxes detected. Try lowering the threshold.")

---
## 📝 Summary

| Model | Role | Input | Output |
|------|------|------|------|
| **Grounding DINO** | Open-set detection | Image + Text labels | Bounding Boxes |
| **SAM** | Promptable segmentation | Image + Box/Point | Pixel Masks |
| **Grounded SAM** | Text→Segmentation | Image + Text labels | Boxes + Masks |

### Key Takeaways

1. **Grounding DINO** can detect any object via a text prompt (open-vocabulary)
2. **SAM** is a foundation model that produces high-quality segmentation masks from various prompts
3. Combining the two enables **pixel-level segmentation from text alone**
4. **Threshold tuning** controls precision vs recall
5. SAM's **3-mask output** is an elegant way to handle ambiguity

### Lab 1 ↔ Lab 2 Connection

| Lab 1 (DINO ViT) | Lab 2 (Grounding DINO + SAM) |
|---|---|
| Self-supervised feature learning | Open-set object detection |
| Image-level representation | Object-level detection + segmentation |
| Attention → implicit localization | Explicit bounding box + mask |
| Classification (linear probing) | Detection + Segmentation |

---
*This notebook was created for Grounding DINO + SAM practice.*